# Capstone Project Debugging

This notebook will be used for testing and debugging of the Python code that the notebook will execute. As the project code grew, my desing philosophy shifted towards moving more of the complex code out of the notebook istelf, such that the notebook becomes an orchestrator that produces results, makes plots, and has descriptive text, but that heavy-lifting and boilerplate code is handled elsewhere.

In [ ]:
import pandas as pd

from pipeline.version_config import VersionConfig
from pipeline.pipeline_run import PipelineRun
from pipeline.factory import PipelineFactory

## Config

No snapshot flags set → `build()` leaves all version counters unchanged.
This run will read from GCS but write nothing back.

In [ ]:
config = VersionConfig.load().build()
run    = PipelineRun(config)
stages = PipelineFactory.validate_current(config)

print('\nScenario:', stages.scenario)
print('raw_version   :', config.raw_version)
print('model_version :', config.model_version)

## First-time Setup: Create the Locked Validation Holdout

**Run once**, before any pipeline scenario. Skip if `splits/validation_ids.json` already
exists in GCS — `DataSplitter` detects the file and errors helpfully if it's missing.

```bash
python scripts/create_validation_set.py        # dry-run (default)
python scripts/create_validation_set.py --yes  # write to GCS
```

The script runs Load → DataPreprocessor → FeatureEngineer, then creates a stratified
holdout using the 18-cell key (vertical × tier × above_baseline).

---
## Main Validation Flow

Runs the full `validate_current` scenario against the locked holdout.

## Stage 1 — DataLoader (GCS)

Reads the parquet snapshot at `config.raw_version`. No BigQuery call.

In [ ]:
stages.loader.run(run)
run.summary()

## Stage 2 — DataPreprocessor

Pivots the long-format poll records into one row per video (via `pivot_snapshots`),
joins channel baselines, and runs structural cleanup via `build_clean_dataset`.
Writes `run.df_clean`.

In [ ]:
stages.preprocessor.run(run)
run.summary()

## Stage 3 — FeatureEngineer

Runs the full feature engineering chain on `run.df_clean` (one row per video).
Computes `above_baseline` target, all ratio/growth features, and categorical encodings.
Writes `run.df_engineered`.

In [ ]:
stages.engineer.run(run)
run.summary()

## Stage 4 — DataSplitter

Loads the locked validation IDs from GCS (`splits/validation_ids.json`).
Filters `run.df_engineered` to produce `df_val`, then stratifies the remaining
rows into `df_train` / `df_test` (80/20). Derives unscaled `X_train`, `X_test`,
`X_val` and corresponding `y_` series.

In [ ]:
stages.splitter.run(run)
run.summary()

## Stage 5 — Scaler

Fits `StandardScaler` on `X_train`, transforms `X_train`, `X_test`, and `X_val`.
Captures `run.X_val_unscaled` so Validator can apply each loaded model's own
historical scaler for a fair apples-to-apples comparison.

In [ ]:
stages.scaler.run(run)
run.summary()

## Stage 6 — ModelLoader

Auto-discovers all `models/<model_version>_*/` directories in GCS and loads each
saved model, scaler, and feature column list into `run.models`.

In [ ]:
stages.model_loader.run(run)
print('\nLoaded models:', list(run.models.keys()))

## Stage 7 — Validator

Evaluates each loaded model on the locked validation set (`X_val_unscaled` +
each model's own historical scaler). Populates `run.results` with AUC, accuracy,
F1, precision, recall, and top feature importances per model.

In [ ]:
stages.validator.run(run)

## Stage 8 — Persist Validation Results to GCS

In [ ]:
stages.validation_results_snapshotter.run(run)
# Appends to gs://maduros-dolce-capstone-data/models/{model_version}/validation_results.jsonl

## Results

This is the **first validation-set evaluation** for this project — there is no prior
validation baseline in `data_analysis.ipynb` (only test-set scores were recorded there).

Cross-check model rank order and approximate AUC range against the test scores
from `data_analysis.ipynb`. Validation AUC will typically be slightly lower than
test AUC since the holdout was never touched during training.

In [ ]:
metric_cols = [
    'roc_auc', 'accuracy',
    'f1_above', 'precision_above', 'recall_above',
    'f1_below', 'precision_below', 'recall_below',
]

df_results = (
    pd.DataFrame(run.results).T
    [metric_cols]
    .astype(float)
    .round(4)
)
df_results.index.name = 'model'

df_results.style.highlight_max(color='#d4edda').format('{:.4f}')

In [ ]:
# Top features for each model — sanity check against data_analysis.ipynb
for name, entry in run.results.items():
    top = entry.get('top_features', [])[:5]
    print(f'\n{name}:')
    for f in top:
        key = 'coefficient' if 'coefficient' in f else 'importance'
        print(f'  {f["feature"]:35s}  {f[key]:+.4f}')